In [1]:
import glob
from joblib import Parallel, delayed
#from catalog_plotter import plot_var, get_array
import numpy as np
import matplotlib.pyplot as plt
import cartopy as cp
import cartopy.crs as ccrs
import plotter_config as pc
import sim_name_remap as map
import netCDF4 as nc4
import os
import datetime as dt
from scipy.interpolate import interp1d
import pickle as pkl
import matplotlib.tri as tri
from plotter import plot_map
import utils

In [2]:

sims = ['Aug2022_E5-1hr_oxSO2update','Aug2022_E5-1hr_oxSO2-0.75','2022_E5-1hr_H2SO4prod-0.75']  # There must be the same number of both these files
files0 = sorted(glob.glob(map.remap[sims[0]]))
files1 = sorted(glob.glob(map.remap[sims[1]]))
files2 = sorted(glob.glob(map.remap[sims[2]]))

date0 = '2022-08-01'
date1 = '2022-08-31'

F0 = [item for item in files0 if date0 in item][0]
find0 = list(files0).index(F0)
F1 = [item for item in files0 if date1 in item][0]
find1 = list(files0).index(F1)
files0 = files0[find0:find1+1]

F0 = [item for item in files1 if date0 in item][0]
find0 = list(files1).index(F0)
F1 = [item for item in files1 if date1 in item][0]
find1 = list(files1).index(F1)
files1 = files1[find0:find1+1]

F0 = [item for item in files2 if date0 in item][0]
find0 = list(files2).index(F0)
F1 = [item for item in files2 if date1 in item][0]
find1 = list(files2).index(F1)
files2 = files2[find0:find1+1]

var_str = 'r_SO2_OH'
lon_bnds = [125,140]
lat_bnds = [25,38]
lev_ind = 13


In [3]:

nc0 = nc4.Dataset(files0[0])

lev = nc0.variables['lev'][:]
lon = nc0.variables['lon'][:]
lat = nc0.variables['lat'][:]

lon_ind0 = np.where(lon >= lon_bnds[0])[0][0]
lon_ind1 = np.where(lon <= lon_bnds[1])[0][-1]+1

lat_ind0 = np.where(lat >= lat_bnds[0])[0][0]
lat_ind1 = np.where(lat <= lat_bnds[1])[0][-1]+1


In [4]:

var0 = []
var1 = []
var2 = []

for f in range(len(files0)):

    nc0 = nc4.Dataset(files0[f])
    nc1 = nc4.Dataset(files1[f])
    nc2 = nc4.Dataset(files2[f])    

    curr_var0 = nc0.variables[var_str][:,lev_ind,lat_ind0:lat_ind1,lon_ind0:lon_ind1].flatten()
    curr_var1 = nc1.variables[var_str][:,lev_ind,lat_ind0:lat_ind1,lon_ind0:lon_ind1].flatten()
    curr_var2 = nc2.variables[var_str][:,lev_ind,lat_ind0:lat_ind1,lon_ind0:lon_ind1].flatten()    

    var0.extend(curr_var0)
    var1.extend(curr_var1)
    var2.extend(curr_var2)    

print(np.mean(var0))
print(np.mean(var1))
print(np.mean(var2))


363.80942
374.4926
355.42575



plt.hist(var0, color='black', label = 'oxSO2update', histtype = 'step')
plt.hist(var1, color='red', label = 'oxSO2update-0.75', histtype='step')

plt.title(var_str + '; ' + str(lon_bnds) + ' E; ' + str(lat_bnds) + ' N; ' + str(round(lev[lev_ind],1)) + ' hPa')

plt.legend()
plt.xscale('log')
